# 4. 세 모델 답 비교하기 — 원본 · 내 모델 · GPT
학습에 쓰지 않은 시험 문제(`test.jsonl`)를 세 모델에 똑같이 묻고 나란히 비교해요.

| 순서 | 하는 일 | 결과 |
|---|---|---|
| 1 | 설정 | — |
| 2 | 세 모델에 묻기 (실행할 때마다 새로 물어요, 1~2분) | `results/compare_날짜시각.json` |
| 3 | 표로 보기 · 자동 체크 | 화면 |
| 4 | 웹 비교표 | `compare_report.html` (브라우저로 열림) |
| 5 | (선택) 블라인드 평가 | `compare_blind.html` |

준비: Ollama에 원본 `qwen2.5:3b`와 내 모델 등록 (`setup_ollama.ps1`, README) · `.env`에 `OPENAI_API_KEY` · 같은 폴더에 `compare_tools.py`

In [9]:
# ▶ 1. 설정 — 여기만 고치세요
import os
import pandas as pd
from dotenv import load_dotenv
from compare_tools import connect, ask_models, auto_summary, make_report

MY_MODEL = 'mbcsports'      # ollama list 에 나온 내 모델 이름
load_dotenv()               # 같은 폴더의 .env 에서 OPENAI_API_KEY 읽기
MODELS = connect(MY_MODEL, os.getenv('OPENAI_API_KEY'))
pd.set_option('display.max_colwidth', None)
print('API 키:', '준비됨' if os.getenv('OPENAI_API_KEY') else '없음 → gpt-4.1-mini 칸은 (키 없음)')

API 키: 준비됨


In [10]:
# ▶ 2. 세 모델에 묻기
test = pd.read_json('test.jsonl', lines=True)
print('지시문:', test['instruction'][0])
answers = ask_models(MODELS, test, 'compare')

지시문: 보도자료를 읽고 기사 첫 문장(리드)을 한 문장으로 쓰세요.
묻는 중: 원본 Qwen2.5-3B (5문제)
묻는 중: 내가 학습한 모델 (5문제)
묻는 중: gpt-4.1-mini (5문제)
✅ 기록 저장: results\compare_20261112_1552.json


In [11]:
# ▶ 3-1. 표로 보기 — 정답과 세 모델 답 전체|
pd.DataFrame({'정답 리드': test['output'], **answers})

,정답 리드,원본 Qwen2.5-3B,내가 학습한 모델,gpt-4.1-mini
0,다음 달 중동 원정에 나서는 축구대표팀 명단이 발표됐습니다. 모레노 감독은 손흥민과 은퇴 소동을 벌인 96년생 3인방 선수들을 모두 발탁해 정면 돌파 의지를 내비쳤는데요. 주장은 손흥민이라고 못을 박았습니다. 박재웅 기자입니다.,"김민재와 황인범, 황희찬이 보수적으로 보이는 리더십 문제를 제기하며 손흥민과 함께 최고의 선수로 선택된 96년생 동기 3인방의 은퇴 소동을 둘러싼 갈등이 모레노 임시 감독의 결정을 더욱 복잡하게 만들었다.",축구대표팀 임시 감독을 맡은 모레노 감독이 손흥민의 주장직을 유지할 것이라는 입장을 밝혔습니다.,모레노 축구대표팀 임시 감독이 손흥민과 96년생 동기 선수들을 모두 발탁하며 주장 손흥민 체제를 유지하고 내부 갈등 해결에 정면 돌파를 선택했다.
1,"프로야구 한국 시리즈에서 삼성이 kt를 꺾고 통산 9번째 우승을 달성했습니다. 정규시즌 2위로 열세가 예상됐지만 1패 뒤 4연승으로 짜릿한 역전 우승을 이루며, 12년 만에 정상에 복귀했는데요. 김태운 기자가 보도합니다.","삼성 라이온즈가 1차전 패배 후 3연승을 이어가며 9부 능선을 넘어서며, 결국 통산 9번째 우승을 차지한 후에야 깨달은 KT의 역전 도전이 실패한 것.","삼성은 지난 2014년 이후 12년 만에 정규시즌 우승을 확정하고, 한국시리즈 MVP에 오른 구자욱의 생애 첫 우승에 환호를 이어갔습니다.",삼성 라이온즈가 1차전 패배 후 3연승을 거두며 12년 만에 통산 9번째 한국시리즈 우승을 확정했다.
2,여자 기계체조 평균대에서 금메달을 따낸 '작은 거인' 황서현 선수가 하루 사이에 말 그대로 깜짝 스타로 떠올랐습니다. 오늘 입국장에서도 가장 큰 주목을 받았던 황서현의 금메달 뒷이야기를 일본 나고야 현지에서 곽승규 기자가 전합니다.,135cm의 작은 체구에도 강단 위에서 너비 10cm의 평균대를 넘어서는 황서현의 강한 자신감과 40년의 투혼으로 이뤄진 금메달 소식이 세상에 알려졌다.,여자기계체조 황서현 선수의 아시안게임 금메달이 화제가 됐습니다.,135cm의 작은 체구에도 불구하고 40년 만에 아시안게임 여자기계체조 금메달을 획득한 황서현이 새로운 스타로 떠올랐다.
3,축구대표팀이 우루과이에 완패한 데 이어 베네수엘라와도 졸전 끝에 득점 없이 비겼는데요. 김민재 선수가 축구화를 집어던지며 화를 낸 뒤 경기 후 작심발언을 쏟아냈습니다. 박재웅 기자입니다.,"김민재가 우루과이전에서 피파 랭킹 48위의 베네수엘라와의 무승부를 보고, 수비 선수들의 부족함을 지적하며 팀의 성장을 위해선 다 함께 노력해야 한다는 입장을 분명히 한 후, 김민재의 불만이 김민재와 손흥민, 황인범 등 선수들의 불만을 불러일으키며 모레노호의 조직력 문제를 드러내는 기사 첫 문장은 ""김민재가 우루과이전에서 베네수엘라와의 무승부를 보고, 수비 선수들의 부족함을 지적하며 팀의 성장을 위해선 다 함께 노력해야 한다는 입장을 분명히 한 후, 선수들의 불만이 불러일으키며 모레노호의 조직력 문제를 드러내는 기사""입니다.","축구대표팀이 우루과이전에 이어 베네수엘라전에서도 무승부를 기록하며, 모레노 감독의 임시감독 시절이 시작되자마자 불만이 쏟아지고 있습니다.",한국 축구대표팀이 우루과이전 완패에 이어 FIFA 랭킹 48위 베네수엘라와 득점 없이 비기며 내부 불화와 조직력 문제를 드러냈다.
4,배드민턴 안세영 선수가 49연승 행진 속에 프랑스오픈 결승에 올랐는데요. 대회 3연패와 함께 시즌 10번째 우승을 눈앞에 두게 됐습니다.,안세영이 준결승에서 세계 9위의 일본 선수 오쿠하라를 2대 0으로 이기고 결승 진출을 확정지었다.,"여자전 세계랭킹 9위 일본의 오쿠하라를 꺾고, 여자 단식 결승에 오른 안세영 선수는 49연승 행진을 이어갔습니다.",안세영이 세계 9위 오쿠하라를 2대 0으로 완파하며 대회 3회 연속 우승과 올 시즌 10번째 우승에 한 걸음 다가섰다.


In [12]:
# ▶ 3-2. 자동 체크 요약 — 형식만 봐요 (참고용)
auto_summary(test, answers)

,답한 문제,끝인사 없음,기자에게 넘김,2문장 이하,길이 비슷(0.5~2배),입력에 없는 숫자 없음,평균 글자 수
모델,,,,,,,
원본 Qwen2.5-3B,5/5,5/5,0/5,4/5,4/5,5/5,126
내가 학습한 모델,5/5,5/5,0/5,5/5,3/5,5/5,61
gpt-4.1-mini,5/5,5/5,0/5,5/5,4/5,5/5,68


In [13]:
# ▶ 4. 웹 비교표 — 원본(회색) · 내 모델(초록) · GPT(파랑) 순서 고정
make_report(test, answers, 'compare_report.html', '같은 문제, 세 모델의 리드 비교')

✅ 비교표 완성: C:\workspace\ai-datajournalism-labs\4_파인튜닝\compare_report.html


In [14]:
# ▶ 5. (선택) 블라인드 평가 — 이름을 가리고 문제마다 A·B·C로 섞기
#    고른 뒤 맨 아래 '🔑 정답지'를 열어 누가 누구였는지 확인해요
make_report(test, answers, 'compare_blind.html', '블라인드 평가: 누가 쓴 리드일까?', blind=True)

✅ 비교표 완성: C:\workspace\ai-datajournalism-labs\4_파인튜닝\compare_blind.html


## 사람이 확인할 것 — 자동 체크가 못 보는 부분
- **사실이 본문과 맞나?** 본문에 없는 원인 · 전망 · 인용 출처를 지어내지 않았나
- **누가 한 말인가?** 인터뷰 발언을 SNS 글이라고 쓰는 식의 바꿔치기
- **말이 자연스러운가?** 어색한 표현, 잘못된 띄어쓰기

⚠ `results/`와 `*.html`에는 기사 내용이 들어 있어요. GitHub에 올리지 않아요.